# LoCED 100M body — 12 layers, dynamic mHC + MagicNorm, GPT-2 vocab — **real HF data** (Kaggle TPU v5e-8 / GPU T4 x2)

Trains the validated **12-layer LoCED** model (dynamic mHC + MagicNorm, GPT-2
vocabulary, sparse top-1 MoE) on **real Hugging Face text** instead of
synthetic tokens, using the pure-JAX reference path (`loced.arch_train`).

| Piece | Params |
|---|---:|
| 12-layer transformer body | **99,749,744** |
| GPT-2 embedding | 12,865,792 |
| GPT-2 output head | 12,865,792 |
| **Full model** | **125,481,328** |
| **Active body per token** | **5,304,176** (5.32 % of body) |

**Model**: `d=256`, 12 physical layers, 24 visits (`12 × full`, then
`6 × reuse + 6 × reindex`), 4 mHC streams with dynamic coefficients,
MagicNorm at every visit exit, 62 routed experts (top-1) + 1 shared expert,
`I=168`, GPT-2 vocabulary (`50,257`).

**Data**: eight datasets, mixed with a deterministic ratio schedule and
tokenised with the **GPT-2** tokenizer (vocab exactly 50,257, so token ids
need no modulo):

| Dataset | Text columns used | Gated |
|---|---|---|
| `nampdn-ai/tiny-textbooks` | `textbook`, `text` | yes |
| `nampdn-ai/tiny-strange-textbooks` | `text` | yes |
| `nampdn-ai/tiny-math-textbooks` | `response` | yes |
| `nampdn-ai/tiny-codes` | `prompt`, `response` | yes |
| `nampdn-ai/tiny-code-textbooks` | `response`, `content` | yes |
| `open-phi/programming_books_llama` | `markdown` | no |
| `nampdn-ai/tiny-webtext` | `human`, `bot` | yes |
| `SciPhi/textbooks-are-all-you-need-lite` | `completion`, `notes` | no |

### Before running

1. Kaggle → **Accelerator: TPU v5e-8 / GPU T4 x2** (8 chips).
2. Kaggle → **Add-ons → Secrets** → add `HF_TOKEN` (a read token for the
   account that accepted the terms of the six gated `nampdn-ai` datasets).
3. Run all cells.

The token is passed to the training subprocess through the environment only;
it is never printed and never written to the repository.

In [ ]:
# ============================================================================
# 1. configuration + locate (or clone) the loced repository
# ============================================================================
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

# "tpu" (Kaggle TPU v5e-8) or "gpu" (Kaggle GPU T4 x2).
ACCELERATOR = os.environ.get("LOCED_ACCELERATOR", "tpu").strip().lower()
MAXTEXT_REF = "maxtext-v0.2.5"
MAXTEXT_REPO = "https://github.com/AI-Hypercomputer/maxtext.git"
# The MaxText extra supplies the right JAX build for the accelerator:
# "tpu" for the TPU wheels, "cuda12" for the CUDA build.
MAXTEXT_EXTRAS = "tpu" if ACCELERATOR == "tpu" else "cuda12"

LOCED_REPO_URL = os.environ.get(
    "LOCED_REPO_URL", "https://github.com/offx-zinth/loced.git")
LOCED_REPO_REF = os.environ.get("LOCED_REPO_REF", "master")

# --- run knobs -------------------------------------------------------------
TRAIN_STEPS = int(os.environ.get("LOCED_TRAIN_STEPS", "2000"))
GLOBAL_BATCH = int(os.environ.get("LOCED_BATCH", "8"))
SEQ_LEN = int(os.environ.get("LOCED_SEQ", "128"))
LR = float(os.environ.get("LOCED_LR", "1e-3"))
SEED = int(os.environ.get("LOCED_SEED", "0"))
TOKENIZER = os.environ.get("LOCED_TOKENIZER", "gpt2")
MAX_EXAMPLES = int(os.environ.get("LOCED_MAX_EXAMPLES", "0")) or None
RUN_GENERATION = os.environ.get("LOCED_RUN_GENERATION", "1") == "1"
EXPECTED_BACKEND = os.environ.get("LOCED_EXPECTED_BACKEND", ACCELERATOR)
# 0 means "do not check the device count" (the GPU kernel may expose 1 or 2
# GPUs depending on the account).
EXPECTED_DEVICES = int(os.environ.get("LOCED_EXPECTED_DEVICES",
                                         "8" if ACCELERATOR == "tpu" else "0"))
TIMEOUT_S = int(os.environ.get("LOCED_TIMEOUT", "21600"))

HF_DATASETS = os.environ.get(
    "LOCED_HF_DATASETS",
    "nampdn-ai/tiny-textbooks,nampdn-ai/tiny-strange-textbooks,"
    "nampdn-ai/tiny-math-textbooks,nampdn-ai/tiny-codes,"
    "nampdn-ai/tiny-code-textbooks,open-phi/programming_books_llama,"
    "nampdn-ai/tiny-webtext,"
    "SciPhi/textbooks-are-all-you-need-lite",
)
HF_RATIOS = os.environ.get("LOCED_HF_RATIOS", "1,1,1,1,1,1,1,1")

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
WORK.mkdir(parents=True, exist_ok=True)
MAXTEXT_DIR = WORK / "maxtext-v0.2.5"
VENV_DIR = WORK / "loced-venv"
PY = VENV_DIR / "bin" / "python"
SCRIPTS = WORK / "loced_scripts"
CACHE = WORK / "jax_cache"
HF_CACHE = WORK / "hf_cache"
PREFETCH_DIR = WORK / "loced_data"
REPORT = WORK / "loced_100m_body_12l_mhc_gpt2_hf_report.json"
TRAIN_LOG = WORK / "arch_train_hf.log"
for _p in (SCRIPTS, CACHE, HF_CACHE, PREFETCH_DIR):
    _p.mkdir(parents=True, exist_ok=True)


def _is_loced_package_dir(path: Path) -> bool:
    """The repository root IS the python package `loced`."""
    return ((path / "arch_train.py").is_file()
            and (path / "model.py").is_file()
            and (path / "maxtext" / "train.py").is_file())


def _find_loced_package_dir():
    candidates = [WORK, WORK / "loced", Path.cwd(), Path.cwd() / "loced"]
    if os.environ.get("LOCED_ROOT"):
        candidates.insert(0, Path(os.environ["LOCED_ROOT"]))
    for root in candidates:
        if _is_loced_package_dir(root):
            return root
        if _is_loced_package_dir(root / "loced"):
            return root / "loced"
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for path in kaggle_input.glob("**/arch_train.py"):
            if _is_loced_package_dir(path.parent):
                return path.parent
    return None


# Keep a reused Kaggle working directory in sync with the remote branch.
_target = WORK / "loced"
if LOCED_REPO_URL and (_target / ".git").is_dir():
    _env = os.environ.copy()
    _env["GIT_TERMINAL_PROMPT"] = "0"
    print(f"updating the existing checkout {_target} ({LOCED_REPO_REF})")
    subprocess.run(["git", "-C", str(_target), "fetch", "--depth", "1",
                    "origin", LOCED_REPO_REF], check=True, env=_env)
    subprocess.run(["git", "-C", str(_target), "reset", "--hard", "FETCH_HEAD"],
                   check=True, env=_env)

PACKAGE_DIR = _find_loced_package_dir()
if PACKAGE_DIR is None:
    if LOCED_REPO_URL:
        if not (_target / ".git").is_dir():
            _env = os.environ.copy()
            _env["GIT_TERMINAL_PROMPT"] = "0"
            print(f"cloning {LOCED_REPO_URL} ({LOCED_REPO_REF}) -> {_target}")
            subprocess.run(["git", "clone", "--depth", "1", "--branch",
                            LOCED_REPO_REF, LOCED_REPO_URL, str(_target)],
                           check=True, env=_env)
        PACKAGE_DIR = _find_loced_package_dir()
if PACKAGE_DIR is None:
    raise RuntimeError(textwrap.dedent("""
        The loced package was not found.
        - Upload the repository (the folder that contains arch_train.py) to
          /kaggle/working/loced, or as a Kaggle Dataset under /kaggle/input,
        - or set LOCED_REPO_URL (and LOCED_REPO_REF) to a branch that
          contains arch_train.py.
    """))

# The repository root IS the package `loced`, so its PARENT has to be on
# PYTHONPATH for `python -m loced.arch_train` to resolve.
LOCED_PARENT = PACKAGE_DIR.parent

print("ACCELERATOR :", ACCELERATOR)
print("WORK        :", WORK)
print("PACKAGE_DIR :", PACKAGE_DIR)
print("PYTHONPATH  :", LOCED_PARENT)
print("MAXTEXT_DIR :", MAXTEXT_DIR)
print("VENV        :", VENV_DIR)
print("STEPS       :", TRAIN_STEPS, " BATCH:", GLOBAL_BATCH, " SEQ:", SEQ_LEN)
print("DATASETS    :", HF_DATASETS)
print("RATIOS      :", HF_RATIOS)
print("REPORT      :", REPORT)

## 2. Python 3.12 + MaxText v0.2.5 (TPU)

MaxText v0.2.5 needs Python >= 3.12, so the notebook creates an isolated
`uv` venv. The install uses MaxText's tested TPU dependency set and pins the
HF data/tokenizer stack used by `loced.hf_data`.

In [ ]:
# ============================================================================
# 2. isolated Python 3.12 venv with the MaxText accelerator extra
#    + datasets + transformers
# ============================================================================
if shutil.which("uv") is None:
    print("installing uv ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv is not on PATH after pip install")

if not PY.exists():
    print("creating the Python 3.12 venv ...")
    subprocess.run([UV, "venv", "--python", "3.12", "--seed", str(VENV_DIR)],
                   check=True)
print(subprocess.check_output([str(PY), "--version"], text=True).strip())

if not (MAXTEXT_DIR / "pyproject.toml").is_file():
    print(f"cloning MaxText {MAXTEXT_REF} ...")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", MAXTEXT_REF,
                    MAXTEXT_REPO, str(MAXTEXT_DIR)], check=True)
else:
    print("MaxText checkout already present")

print(f"installing maxtext[{MAXTEXT_EXTRAS}] (resolution=lowest) ...")
subprocess.run([UV, "pip", "install", "--python", str(PY), "--link-mode=copy",
                "-e", f".[{MAXTEXT_EXTRAS}]", "--resolution=lowest"],
               cwd=str(MAXTEXT_DIR), check=True)

print("installing the HF data / tokenizer stack ...")
subprocess.run([UV, "pip", "install", "--python", str(PY), "--link-mode=copy",
                "datasets>=2.19,<4", "transformers>=4.40,<5",
                "huggingface_hub>=0.23,<1"], check=True)
subprocess.run([UV, "pip", "check", "--python", str(PY)], check=False)

print(subprocess.check_output(
    [str(PY), "-c",
     "import sys, jax, datasets, transformers; "
     "print(sys.version); print('jax', jax.__version__); "
     "print('datasets', datasets.__version__); "
     "print('transformers', transformers.__version__)"],
    text=True,
))

## 3. Hugging Face token + gated-dataset access

The token is read from the `HF_TOKEN` Kaggle secret (falling back to the
environment). It is passed only through subprocess environments and is never
printed.

Six of the eight datasets are **gated**: the token's owner must have accepted
their terms on the Hugging Face website, otherwise the load fails. A dataset
that cannot be loaded is reported and skipped — it never aborts the run.

In [ ]:
# ============================================================================
# 3. HF_TOKEN from the Kaggle secret (or the environment)
# ============================================================================
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretClient
        HF_TOKEN = UserSecretClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = ""

if HF_TOKEN:
    print("HF token loaded (the value is not printed).")
    print("Note: the six gated nampdn-ai datasets still need their terms "
          "accepted for this account.")
else:
    print("=" * 70)
    print("HF_TOKEN is missing.")
    print("Add it in Kaggle: Add-ons -> Secrets -> New Secret -> name "
          "'HF_TOKEN' -> paste your Hugging Face read token, then re-run "
          "this notebook.")
    print("The six gated nampdn-ai datasets (tiny-textbooks, "
          "tiny-strange-textbooks, tiny-math-textbooks, tiny-codes, "
          "tiny-code-textbooks, tiny-webtext) additionally require that the "
          "terms are accepted for your account on the dataset page "
          "(https://huggingface.co/datasets/<name> -> 'Agree and access "
          "granted').")
    print("Without a token the run continues with the two public datasets "
          "(open-phi/programming_books_llama and "
          "SciPhi/textbooks-are-all-you-need-lite); the gated ones are "
          "skipped with a warning.")
    print("=" * 70)

os.environ["HF_TOKEN"] = HF_TOKEN

## 4. Train on real HF data (TPU v5e-8)

Runs `python -m loced.arch_train --data hf`:

parameter audit → sparse top-1 MoE dispatch (with the dense-reference parity
check) → mesh gradient check → streaming real-data training → evaluation at
`reuse_p` 0 / 0.5 / 1.0 → greedy generation → JSON report.

The output is streamed back into this notebook and tee'd to
`/kaggle/working/arch_train_hf.log`.

In [ ]:
# ============================================================================
# 4. run the 12-layer training on the real datasets
# ============================================================================
BASE_ENV = os.environ.copy()
BASE_ENV.update({
    "PYTHONPATH": f"{MAXTEXT_DIR / 'src'}:{LOCED_PARENT}",
    "JAX_COMPILATION_CACHE_DIR": str(CACHE),
    "XLA_PYTHON_CLIENT_MEM_FRACTION": "0.90",
    "XLA_PYTHON_CLIENT_PREALLOCATE": "true",
    "TOKENIZERS_PARALLELISM": "false",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "HF_HUB_DOWNLOAD_TIMEOUT": "120",
    "HF_HUB_ETAG_TIMEOUT": "60",
    "HF_HOME": str(HF_CACHE),
    "HF_DATASETS_CACHE": str(HF_CACHE / "datasets"),
    "HF_HUB_CACHE": str(HF_CACHE / "hub"),
    "HF_XET_CACHE": str(HF_CACHE / "xet"),
    "PYTHONDONTWRITEBYTECODE": "1",
    "TF_CPP_MIN_LOG_LEVEL": "2",
    "LOCED_ROOT": str(PACKAGE_DIR),
    "LOCED_ACCELERATOR": ACCELERATOR,
    "LOCED_WORK": str(WORK),
})
# Kaggle/Colab export MPLBACKEND=module://matplotlib_inline.backend_inline for
# the notebook process; the venv has no matplotlib_inline, so importing
# matplotlib there fails. This notebook needs no plotting at all, but the
# override keeps the subprocess safe.
BASE_ENV["MPLBACKEND"] = "Agg"

# jax-cuda12 loads the pip-installed NVIDIA libraries with ctypes.  In a
# venv they live under site-packages/nvidia/*/lib, which is not on the
# default loader path, so cuSPARSE/cuDNN can otherwise be "not found".
_nvidia_code = (
    "import site; from pathlib import Path; "
    "print(':'.join(str(p) for r in site.getsitepackages() "
    "for p in sorted((Path(r) / 'nvidia').glob('*/lib')) if p.is_dir()))"
)
_nvidia_libs = subprocess.check_output(
    [str(PY), "-c", _nvidia_code], text=True).strip()
if _nvidia_libs:
    BASE_ENV["LD_LIBRARY_PATH"] = ":".join(
        x for x in (_nvidia_libs, os.environ.get("LD_LIBRARY_PATH", "")) if x
    )
print("NVIDIA lib dirs:", len(_nvidia_libs.split(":")) if _nvidia_libs else 0)

cmd = [
    str(PY), "-m", "loced.arch_train",
    "--data", "hf",
    "--datasets", HF_DATASETS,
    "--ratios", HF_RATIOS,
    "--tokenizer", TOKENIZER,
    "--seq-len", str(SEQ_LEN),
    "--batch-size", str(GLOBAL_BATCH),
    "--steps", str(TRAIN_STEPS),
    "--lr", str(LR),
    "--seed", str(SEED),
    "--out", str(REPORT),
]
if MAX_EXAMPLES:
    cmd += ["--max-examples", str(MAX_EXAMPLES)]
if not RUN_GENERATION:
    cmd += ["--no-run-generation"]
if EXPECTED_BACKEND:
    cmd += ["--expected-backend", EXPECTED_BACKEND]
if EXPECTED_DEVICES:
    cmd += ["--expected-devices", str(EXPECTED_DEVICES)]

env = BASE_ENV.copy()
env["HF_TOKEN"] = HF_TOKEN
env["PYTHONUNBUFFERED"] = "1"

print("running:", " ".join(cmd))
print("cwd     :", LOCED_PARENT)
t0 = time.time()
log = open(TRAIN_LOG, "w")
proc = subprocess.Popen(cmd, cwd=str(LOCED_PARENT), env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
        if time.time() - t0 > TIMEOUT_S:
            proc.kill()
            raise TimeoutError(f"the training exceeded {TIMEOUT_S}s")
    rc = proc.wait()
finally:
    log.close()
elapsed = time.time() - t0
print(f"[train] exit={rc} elapsed={elapsed / 60:.1f} min")
if rc != 0:
    raise RuntimeError(f"the training failed (exit {rc}); see the log above")

## 5. Report

The JSON report holds the parameter audit, the MoE dispatch parity check, the
mesh gradient check, the selection budget, the full training history, the
`reuse_p` sweep and the greedy generation. No plot is written.

In [ ]:
# ============================================================================
# 5. final report
# ============================================================================
print("report   :", REPORT)
print("log      :", TRAIN_LOG)
if not REPORT.is_file():
    raise RuntimeError(f"no report at {REPORT}; the training did not finish")

report = json.loads(REPORT.read_text())
print("\n================ REPORT ================")
print("backend            :", report["backend"], " devices:", report["devices"])
print("data               :", report["data"]["source"],
      report["data"].get("datasets", ""))
print("moe dispatch       :", report["moe_dispatch"],
      report.get("moe_dispatch_parity"))
print("mesh grad check    :", report["mesh_gradient_check"])
print("selection          :", report["params"]["selection"])
p = report["params"]
print(f"total              : {p['total']:,} ({p['total'] / 1e6:.3f} M)")
print(f"  body             : {p['body']:,} ({p['body'] / 1e6:.3f} M)")
print(f"  embedding+head   : {p['embedding'] + p['head']:,}")
print(f"active body/token  : {p['active_body_per_token']:,} "
      f"({p['active_body_per_token'] / 1e6:.3f} M, "
      f"{p['body_active_fraction']:.2%} of body)")
print("sinkhorn max err   :", p["sinkhorn_max_error"])
t = report["training"]
print(f"steps              : {t['steps']}  batch {t['global_batch']} x "
      f"{t['seq_len']} tokens")
print(f"ms/step (mean)     : {t['mean_ms_per_step_after_first']:.1f}")
print(f"loss first -> last : {t['history'][0]['loss']:.4f} -> "
      f"{t['history'][-1]['loss']:.4f}")
for k, v in report["eval"].items():
    print(f"eval reuse_p={k}: loss {v['loss']:.4f}  ce {v['ce']:.4f}  "
          f"acc {v['next_token_acc']:.3f}")
print("oracle-vs-indexer  :", report["eval_oracle_vs_indexer_spread"])
if report.get("generation"):
    print("generated ids      :", report["generation"]["generated_ids"])

## 6. Notes

- The training path is the validated one: a 1-D `data` mesh, replicated
  parameters, a batch sharded over the data axis, Muon via
  `loced.train.make_optimizer`, and the `reuse_p` ramp (0 for the first 20 % of
  the steps, ramped to 1 over the next 30 %, then 1).
- Inside `shard_map` the gradient of a replicated parameter is already
  all-reduced, so the local loss is divided by the device count and the
  gradients are **not** `pmean`ed.
- The MoE load-balance auxiliary loss uses **global** (all-reduced) batch
  statistics in both the sparse and the dense path.
- The selection budget is asserted: at `seq_len=128` there are 96 candidates
  and the budget is 64, so the sparse path is genuinely exercised.
- The token is only ever passed through the subprocess environment.